# 03 — ELF Continuous Flow-Matching

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/winstonsmith1897/DantinoX/blob/main/docs/notebooks/03_elf_flow_matching.ipynb)

This notebook trains an **ELF (Embedded Language Flow)** model — a continuous flow-matching language model that operates in a learned embedding space.

**Key concepts:**
- `ELFTransformer` — bidirectional transformer with in-context time and CFG control tokens
- Forward process: `z_t = t·x + (1−t)·ε`, interpolates between noise (t=0) and data (t=1)
- Logit-normal time schedule during training, uniform linear schedule at inference
- Classifier-free guidance (CFG) via dedicated control tokens
- Euler ODE sampler for deterministic generation, SDE sampler (γ > 0) for stochastic

In [ ]:
!pip install -q git+https://github.com/winstonsmith1897/DantinoX.git#egg=dantinox[all]

## 1. Data

In [ ]:
import urllib.request, os
if not os.path.exists('tiny_shakespeare.txt'):
    urllib.request.urlretrieve(
        'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt',
        'tiny_shakespeare.txt'
    )
    print('Downloaded tiny_shakespeare.txt')

with open('tiny_shakespeare.txt') as f:
    text = f.read()
print(f'Corpus: {len(text):,} characters')

## 2. Model configuration

ELF uses a **separate `ELFConfig`** that specifies the continuous embedding space (`embed_dim`) and the transformer backbone (`model_dim`). The two need not be equal — `embed_dim` is the flow space, `model_dim` is the attention dimension.

In [ ]:
from dantinox.core.config import ELFConfig
from dantinox.core.elf import ELFTransformer
from flax import nnx

config = ELFConfig(
    embed_dim   = 256,   # continuous flow-space dimension
    bottleneck_dim = 64, # bottleneck between embed and model space
    model_dim   = 256,   # transformer hidden dim
    n_heads     = 4,
    head_size   = 64,
    num_blocks  = 4,
    vocab_size  = 200,   # small for demo
    max_seq_len = 128,
    dropout     = 0.0,
    gradient_checkpointing = False,
)

model = ELFTransformer(config, rngs=nnx.Rngs(42))
params = sum(x.size for x in __import__('jax').tree_util.tree_leaves(
    nnx.state(model, nnx.Param)))
print(f'ELFTransformer — {params / 1e6:.2f}M parameters')

## 3. Forward process

ELF's forward process interpolates linearly between noise ε ~ N(0,I) and clean embeddings x:

```
z_t = t·x + (1−t)·ε     t ∈ [0, 1]
```

- At **t=0**: z₀ = ε (pure noise)
- At **t=1**: z₁ = x (clean embedding)

The network is trained to predict the clean embedding x̂ from z_t (x-prediction objective).

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt

# Visualise the interpolation at different t values
rng = jax.random.PRNGKey(0)
eps = jax.random.normal(rng, (256,))     # noise
x   = jnp.ones(256) * 0.5               # clean embedding (constant for illustration)

ts  = np.linspace(0, 1, 6)
fig, axes = plt.subplots(1, 6, figsize=(14, 2.5))
for ax, t in zip(axes, ts):
    z_t = t * x + (1 - t) * eps
    ax.hist(np.array(z_t), bins=20, color='steelblue', alpha=0.8)
    ax.set_title(f't = {t:.1f}')
    ax.set_xlim(-3, 3)
    ax.set_xlabel('z_t value')
axes[0].set_ylabel('count')
plt.suptitle('ELF forward process: z_t = t·x + (1−t)·ε', y=1.02)
plt.tight_layout()
plt.show()

## 4. Time schedule

Training uses a **logit-normal** time schedule that concentrates samples near t≈0.18 (noisy regime), ensuring the model is well-trained where the denoising task is hardest.
At inference a uniform linear schedule is used instead.

In [ ]:
from dantinox.core.diffusion import logit_normal_schedule

# Training schedule vs inference schedule
train_ts  = logit_normal_schedule(64, pmean=-1.5, pstd=0.8)
infer_ts  = jnp.linspace(0.0, 1.0, 65)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.5))
ax1.hist(np.array(train_ts), bins=30, color='coral', alpha=0.8, label='logit-normal (train)')
ax1.set_xlabel('t')
ax1.set_title('Training time schedule')
ax1.legend()

ax2.plot(np.array(infer_ts), marker='.', color='steelblue', label='uniform (inference)')
ax2.set_xlabel('step')
ax2.set_ylabel('t')
ax2.set_title('Inference ODE steps')
ax2.legend()
plt.tight_layout()
plt.show()

## 5. Training

Training an ELF model uses the same `Trainer` as AR and Diffusion — only the config changes.

⚠️ **Note:** ELF requires significantly more training steps than AR or Diffusion to produce coherent output. On a free Colab T4 this demo trains for 1 epoch as a proof-of-concept.

In [ ]:
import dantinox as dx

elf_cfg = dx.ELFConfig(
    embed_dim      = 256,
    bottleneck_dim = 64,
    model_dim      = 256,
    n_heads        = 4,
    head_size      = 64,
    num_blocks     = 4,
    vocab_size     = 200,
    max_seq_len    = 128,
    elf_n_steps    = 32,
    elf_cfg_scale  = 1.5,
)

train_cfg = dx.TrainingConfig(
    lr         = 1e-3,
    epochs     = 1,
    batch_size = 16,
)

paradigm = dx.ContinuousParadigm(elf_cfg)
run_dir  = dx.Trainer(paradigm, train_cfg).fit('tiny_shakespeare.txt')
print('Checkpoint saved to:', run_dir)

## 6. Generation

ELF generation denoises from pure Gaussian noise to clean token embeddings using an Euler ODE, then decodes via the shared unembedding head.

```
z_0 ~ N(0, I)   →   ODE steps (t: 0→1)   →   argmax(W_e^T z_1)   →   tokens
```

In [ ]:
from dantinox.core.generation import elf_generate
from flax import nnx

# Load model from saved checkpoint
elf_model = dx.load(run_dir, paradigm=paradigm)

# Generate 4 samples of 32 tokens
tokens = elf_generate(
    elf_model,
    gen_len    = 32,
    batch_size = 4,
    n_steps    = 32,
    cfg_scale  = elf_cfg.elf_cfg_scale,
    seed       = 42,
)
print('Generated token IDs:')
print(tokens)

## 7. CFG guidance scale

Classifier-Free Guidance (CFG) with scale `w > 1` steers generation towards higher-confidence tokens. The model is called twice — once conditioned and once unconditioned — and the outputs are combined:

```
logit = logit_uncond + w * (logit_cond - logit_uncond)
```

Lower `w` (≈1) → more diverse. Higher `w` (≈5) → more focused but potentially less coherent.

In [ ]:
import numpy as np

# Compare generation at different CFG scales
cfg_scales = [1.0, 1.5, 3.0, 5.0]
all_tokens = []
for w in cfg_scales:
    toks = elf_generate(elf_model, gen_len=32, batch_size=1,
                        n_steps=32, cfg_scale=w, seed=42)
    all_tokens.append(toks[0].tolist())
    print(f'cfg_scale={w:.1f}  unique_tokens={len(set(toks[0].tolist()))}  '
          f'sample={toks[0, :10].tolist()}')